# Prepocesing - Hasil penghapusan Outlier

Pada prepocesing ini saya menggunakan library pyod untuk penghapusan oulier, dimana saya menggunakan 3 metode yaitu:
- KNN
- Abod
- Lof

In [1]:
!pip install pyod


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python3 -m pip install --upgrade pip


Dataset yang digunakan adalah Iris dataset dimana saya mengambil bawaan dari scikit-learn. Dataset ini berisi 150 sampel bunga Iris dengan 4 fitur numerik: panjang dan lebar sepal, serta panjang dan lebar petal. Dataset juga memiliki kolom target yang menunjukkan spesies bunga (setosa, versicolor, virginica).

Dimana yang pertama ini saya Memuat dataset dimana dataset dimuat ke dalam DataFrame pandas agar mudah diolah. yang kedua saya mengubah nama kolom, misalnya "sepal length (cm)" menjadi "sepal_length". yang ketiga saya menambahkan kolom target variety dimana kolom target asli (target) diubah menjadi nama spesies bunga, sehingga lebih mudah dibaca. yang keempat saya menghapus kolom target asli: Agar tidak redundan dan memudahkan analisis. dan yang terakhir adalah mengecek data dimana dicek ukuran dataset dan ditampilkan beberapa baris pertama untuk memastikan data sudah benar.


In [8]:
# --- Import library utama ---
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from pyod.models.knn import KNN
from pyod.models.lof import LOF
from pyod.models.abod import ABOD
import warnings
warnings.filterwarnings('ignore')

# --- Load dataset Iris ---
iris = load_iris(as_frame=True)
df = iris.frame
df.rename(columns={
    "sepal length (cm)": "sepal_length",
    "sepal width (cm)": "sepal_width",
    "petal length (cm)": "petal_length",
    "petal width (cm)": "petal_width"
}, inplace=True)
df["variety"] = df["target"].map(dict(enumerate(iris.target_names)))
df.drop(columns="target", inplace=True)

print("Dataset berhasil dimuat:", df.shape)
df.head()

Dataset berhasil dimuat: (150, 5)


,sepal_length,sepal_width,petal_length,petal_width,variety
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


Pada tahap ini, hanya fitur yang numerik yang akan diambil dari dataset Iris, yaitu: sepal_length, sepal_width, petal_length, petal_width.

Dataset numerik ini yang tampil dibawah berisi 150 sampel dengan 4 kolom

In [11]:
# --- Ambil hanya fitur numerik ---
X = df.select_dtypes("number")
print("\n---Fitur Numerik---")
print("="*60)
print(X)


---Fitur Numerik---
     sepal_length  sepal_width  petal_length  petal_width
0             5.1          3.5           1.4          0.2
1             4.9          3.0           1.4          0.2
2             4.7          3.2           1.3          0.2
3             4.6          3.1           1.5          0.2
4             5.0          3.6           1.4          0.2
..            ...          ...           ...          ...
145           6.7          3.0           5.2          2.3
146           6.3          2.5           5.0          1.9
147           6.5          3.0           5.2          2.0
148           6.2          3.4           5.4          2.3
149           5.9          3.0           5.1          1.8

[150 rows x 4 columns]


Pada tahap ini saya menggunakan tiga metode deteksi outlier untuk membersihkan dataset Iris:

- KNN (K-Nearest Neighbors)
- ABOD (Angle-Based Outlier Detection)
- LOF (Local Outlier Factor)


Dan untuk haasil penghapusan outliernya adalah 

- Dataset aslinya ada 150 baris × 5 kolom
- Setelah KNN, ABOD, atau LOF: 135 baris × 5 kolom

Kolomnya tetap sama, hanya saja baris outliernya yang dihapus.

In [13]:
from pyod.models.knn import KNN
from pyod.models.lof import LOF
from pyod.models.abod import ABOD
import pandas as pd

# Ambil hanya fitur numerik
X = df.select_dtypes("number")

# --- Metode 1: KNN ---
knn = KNN()
knn.fit(X)
mask_knn = knn.labels_ == 0
df_knn_clean = df[mask_knn]
df_knn_outliers = df[~mask_knn]  # Data yang terdeteksi sebagai outlier

print("\n--- KNN ---")
print(f"Jumlah outlier: {df_knn_outliers.shape[0]} sampel")
display(df_knn_outliers.head())  # Tampilkan 5 baris outlier pertama

# --- Metode 2: ABOD (Angle-Based Outlier Detection) ---
try:
    abod = ABOD(method='fast')
    abod.fit(X)
    mask_abod = abod.labels_ == 0
    df_abod_clean = df[mask_abod]
    df_abod_outliers = df[~mask_abod]
    
    print("\n--- ABOD ---")
    print(f"Jumlah outlier: {df_abod_outliers.shape[0]} sampel")
    display(df_abod_outliers.head())
except Exception as e:
    print("ABOD error di Python 3.12:", e)
    df_abod_clean = df.copy()
    df_abod_outliers = pd.DataFrame()
    
# --- Metode 3: LOF (Local Outlier Factor) ---
lof = LOF()
lof.fit(X)
mask_lof = lof.labels_ == 0
df_lof_clean = df[mask_lof]
df_lof_outliers = df[~mask_lof]

print("\n--- LOF ---")
print(f"Jumlah outlier: {df_lof_outliers.shape[0]} sampel")
display(df_lof_outliers.head())

# --- Ringkasan Ukuran Data ---
print("\n--- Ringkasan Ukuran Data ---")
print("Ukuran data asli:", df.shape)
print("Setelah hapus outlier (KNN):", df_knn_clean.shape)
print("Setelah hapus outlier (ABOD):", df_abod_clean.shape)
print("Setelah hapus outlier (LOF):", df_lof_clean.shape)



--- KNN ---
Jumlah outlier: 15 sampel


,sepal_length,sepal_width,petal_length,petal_width,variety
41,4.5,2.3,1.3,0.3,setosa
57,4.9,2.4,3.3,1.0,versicolor
60,5.0,2.0,3.5,1.0,versicolor
68,6.2,2.2,4.5,1.5,versicolor
93,5.0,2.3,3.3,1.0,versicolor



--- ABOD ---
Jumlah outlier: 15 sampel


,sepal_length,sepal_width,petal_length,petal_width,variety
15,5.7,4.4,1.5,0.4,setosa
22,4.6,3.6,1.0,0.2,setosa
41,4.5,2.3,1.3,0.3,setosa
59,5.2,2.7,3.9,1.4,versicolor
62,6.0,2.2,4.0,1.0,versicolor



--- LOF ---
Jumlah outlier: 15 sampel


,sepal_length,sepal_width,petal_length,petal_width,variety
13,4.3,3.0,1.1,0.1,setosa
14,5.8,4.0,1.2,0.2,setosa
15,5.7,4.4,1.5,0.4,setosa
33,5.5,4.2,1.4,0.2,setosa
41,4.5,2.3,1.3,0.3,setosa



--- Ringkasan Ukuran Data ---
Ukuran data asli: (150, 5)
Setelah hapus outlier (KNN): (135, 5)
Setelah hapus outlier (ABOD): (135, 5)
Setelah hapus outlier (LOF): (135, 5)
